# CinePal — GPU Embedding on Colab (stage 2)

Stage-2 of the ingestion pipeline. Downloads the cleaned TMDB snapshot pinned in
`configs/default.yaml` under `ingestion.artifacts.snapshot`, splits it into
main/mini/eval, embeds on a free Colab GPU, and publishes three timestamped
parquet files under `embeddings/` in the HF dataset repo.

Stage 1 (the actual TMDB scrape) runs **locally** via:

```bash
python -m dataset.scraper --upload   # produces snapshots/snapshot_YYYYMMDD.parquet
```

Paste the printed snapshot path into `configs/default.yaml` before opening
this notebook. Local ingest then loads the stage-2 outputs with no further
changes:

```bash
python -m db.ingest             # mini
python -m db.ingest --set main  # full set
```

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Click the 🔑 *Secrets* icon (left sidebar) and add:
   - `HF_TOKEN` — huggingface.co → Settings → Access Tokens (write scope)
   - `GITHUB_TOKEN` *(only if the repo is private)* — a PAT with `repo` scope
3. Fill in `REPO_URL` in cell 3.
4. Export YouTube cookies and upload `youtube_cookies.txt` — see the **YouTube cookies** cell.

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# 1 — Verify GPU is available
!nvidia-smi

In [ ]:
# 2 — Install project dependencies
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q sentence-transformers pyarrow pandas numpy huggingface_hub pyyaml python-dotenv tqdm psycopg[binary] pgvector annotated-types pydantic pydantic-settings yt-dlp open-clip-torch Pillow

In [ ]:
# 3 — Clone the repo and add it to the Python path
import os
import sys
from google.colab import userdata

REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/<REPO_NAME>.git"  # <-- fill in before running

# For private repos, inject the GitHub token into the clone URL.
try:
    github_token = userdata.get("GITHUB_TOKEN")
    if github_token:
        REPO_URL = REPO_URL.replace("https://", f"https://{github_token}@")
except Exception:
    pass  # secret not set — assume public repo

if not os.path.isdir("cantucci"):
    os.system(f"git clone {REPO_URL} cantucci")
else:
    os.system("git -C cantucci pull --ff-only")

os.chdir("cantucci")
if "." not in sys.path:
    sys.path.insert(0, ".")

print("Working directory:", os.getcwd())

In [ ]:
# 4 — Surface the required secret as an env var so backend.settings can read it
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as exc:
    raise RuntimeError("Missing Colab secret: HF_TOKEN") from exc
print("Secrets loaded.")

## YouTube cookies (required for trailer download)

YouTube requires authentication to confirm you are not a bot. Without cookies,
yt-dlp will fail with *"Sign in to confirm you're not a bot"* and every trailer
row will be skipped (zero-vector).

**Steps:**
1. In your browser, go to [youtube.com](https://www.youtube.com) and **make sure you are logged in**.
2. Install the [Get cookies.txt LOCALLY](https://chrome.google.com/webstore/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc) extension (Chrome/Edge) or equivalent.
3. Click the extension icon while on youtube.com → export cookies for **youtube.com** → save the file as `youtube_cookies.txt`.
4. In Colab, open the **Files** panel (folder icon in the left sidebar), navigate to the `cantucci/` directory, and upload `youtube_cookies.txt` there.

The next cell will raise an error if the file is missing.

In [ ]:
# 5 — Verify YouTube cookies file
from pathlib import Path

YT_COOKIES = Path("youtube_cookies.txt")
if not YT_COOKIES.exists():
    raise FileNotFoundError(
        "Upload youtube_cookies.txt to the cantucci/ working directory "
        "(see the YouTube cookies cell above) before running the trailer-embed step."
    )
print(f"Cookies file found: {YT_COOKIES.resolve()}")

In [ ]:
# 6 — Load project settings from configs/default.yaml
from backend.settings import get_settings, ARTIFACTS_DIR

cfg = get_settings()
print(
    f"Config loaded — embedding model: {cfg.representation.model}, "
    f"dim: {cfg.representation.embedding_dim}, split seed: {cfg.split.seed}, "
    f"mini_size: {cfg.split.mini_size}"
)

In [ ]:
# 7 — Download the cleaned snapshot from HF, then split
import pandas as pd
from dataset.hub.fetch import fetch_artifact
from dataset.transform import split

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

snapshot_path = fetch_artifact(cfg.ingestion.hf_repo, cfg.ingestion.artifacts.snapshot, token=os.environ["HF_TOKEN"])
df = pd.read_parquet(snapshot_path)
print(f"Loaded snapshot rows: {len(df)}  ({cfg.ingestion.artifacts.snapshot})")

main_df, mini_df, eval_df = split.three_way(
    df,
    mini_size=cfg.split.mini_size,
    eval_frac=cfg.split.eval_frac,
    seed=cfg.split.seed,
)
print(f"Splits — main: {len(main_df)}, mini: {len(mini_df)}, eval: {len(eval_df)}")

In [ ]:
# 8 — Embed on GPU
import gc
import torch
from core.text_encoder import encode_all

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {device}")

main_emb = encode_all(list(main_df["composite_text"]), batch_size=64)
mini_emb = encode_all(list(mini_df["composite_text"]), batch_size=64)
eval_emb = encode_all(list(eval_df["composite_text"]), batch_size=64)

for name, split_df, emb in [("main", main_df, main_emb), ("mini", mini_df, mini_emb), ("eval", eval_df, eval_emb)]:
    assert emb.shape == (len(split_df), cfg.representation.embedding_dim), f"{name}: unexpected shape {emb.shape}"
    print(f"{name}: {emb.shape}")

In [ ]:
# 9 — Embed trailer frames (open_clip ViT-H/14, native 1024-dim)
from dataset.embed.trailer import encode_trailers

gc.collect()
torch.cuda.empty_cache()

main_trailer_emb = encode_trailers(
    list(zip(main_df["id"], main_df["trailer_youtube_key"])),
    cookiefile=str(YT_COOKIES),
)
mini_trailer_emb = encode_trailers(
    list(zip(mini_df["id"], mini_df["trailer_youtube_key"])),
    cookiefile=str(YT_COOKIES),
)
eval_trailer_emb = encode_trailers(
    list(zip(eval_df["id"], eval_df["trailer_youtube_key"])),
    cookiefile=str(YT_COOKIES),
)

for name, split_df, emb in [
    ("main", main_df, main_trailer_emb),
    ("mini", mini_df, mini_trailer_emb),
    ("eval", eval_df, eval_trailer_emb),
]:
    assert emb.shape == (len(split_df), cfg.representation.embedding_dim), f"{name}: unexpected trailer shape {emb.shape}"
    n_nonzero = int((emb.sum(axis=1) != 0).sum())
    print(f"{name}: {emb.shape}  ({n_nonzero} / {len(split_df)} with trailer embedding)")

In [ ]:
# 10 — Upload timestamped artifacts to Hugging Face
from dataset.hub.upload import upload_artifacts

files = upload_artifacts(
    main_df, mini_df, eval_df,
    main_emb, mini_emb, eval_emb,
    main_trailer_emb=main_trailer_emb,
    mini_trailer_emb=mini_trailer_emb,
    eval_trailer_emb=eval_trailer_emb,
    repo_id=cfg.ingestion.hf_repo,
    artifacts_dir=ARTIFACTS_DIR,
)

print("\nPublished. Paste these into configs/default.yaml under `ingestion.artifacts`:\n")
for split_name, filename in files.items():
    print(f"  {split_name}: {filename}")

## Done

1. Copy the paths printed above into `configs/default.yaml`:

   ```yaml
   ingestion:
     hf_repo: "446f6e6e79/CinePal-embeddings"
     artifacts:
       snapshot: snapshots/snapshot_YYYYMMDD.parquet   # set by `python -m dataset.scraper --upload`
       main: embeddings/main_YYYYMMDD.parquet
       mini: embeddings/mini_YYYYMMDD.parquet
       eval_holdout: embeddings/eval_holdout_YYYYMMDD.parquet
   ```

2. Teammates ingest without a GPU:

   ```bash
   python -m db.ingest             # mini
   python -m db.ingest --set main  # full set
   ```

> **Note on GPU vs CPU floating-point:** embedding outputs may differ by ~1e-6 between
> devices. This is below any meaningful threshold for cosine similarity and can be ignored.